# Lecture 11 · Notebook 1 — The KV cache: a memory calculator, GQA, and why we cache at all
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_1_kv_cache_and_gqa.ipynb)

**Companion to [Chapter 8 of the video lecture](https://nlp-course-cuhksz.github.io/lecture11.html#p8-s117)** (the KV cache, slides 116–120) and to
[Chapter 2](https://nlp-course-cuhksz.github.io/lecture11.html#p2-s31) (sharing keys and values: MQA, GQA and MLA, slides 31–33).

During generation a Transformer keeps the keys and values of every past token, in every layer: the **KV cache**. The lecture's formula is

$$\text{KV bytes per token} = 2 \times \text{layers} \times \text{KV heads} \times \text{head dim} \times \text{bytes per number}$$

In this notebook you will:

1. reproduce the lecture's numbers (LLaMA-2-70B: 10.7 GB vs. 1.3 GB for one 4K sequence; OPT-13B: about 800 KB per token),
2. read the configs of real open models from the Hugging Face Hub and see what GQA and MLA save,
3. work out how many users fit on one GPU,
4. check the formula against the real cache of a small model (SmolLM2-135M, which uses GQA),
5. time generation with and without the cache, and
6. turn GPT-2's multi-head attention into grouped-query attention by **mean-pooling heads** and measure what it costs.

Runs on **CPU in about 5–10 minutes** (downloads: SmolLM2-135M, ~270 MB, and GPT-2, ~550 MB).

In [ ]:
!pip -q install -U transformers

In [ ]:
import json, time, copy, torch
from huggingface_hub import hf_hub_download

import matplotlib.pyplot as plt
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
                     "grid.linewidth": 0.8, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "font.size": 11, "lines.linewidth": 2})

torch.manual_seed(0)

def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_value=2):
    """2 (one K and one V) x layers x KV heads x head dim x bytes per number (2 for FP16/BF16)."""
    return 2 * layers * kv_heads * head_dim * bytes_per_value

## 1 · The lecture's numbers
LLaMA-2-70B has 80 layers and 64 query heads of size 128. With plain multi-head attention (MHA) every query head would need its own
key and value head. The real model uses **GQA with 8 KV heads** (slide 117). OPT-13B (slide 118) uses MHA with 40 heads of size 128.
Note that the slides mix decimal units (GB) and binary units (KiB = 1024 bytes); we print both where it matters.

In [ ]:
mha = kv_bytes_per_token(layers=80, kv_heads=64, head_dim=128)
gqa = kv_bytes_per_token(layers=80, kv_heads=8, head_dim=128)
print(f"LLaMA-2-70B per token : MHA {mha / 1e6:.2f} MB   GQA-8 {gqa / 1e6:.2f} MB")
print(f"one 4,096-token request: MHA {4096 * mha / 1e9:.1f} GB   GQA-8 {4096 * gqa / 1e9:.2f} GB")
print(f"batch of 32 x 4K tokens with GQA-8: {32 * 4096 * gqa / 1e9:.0f} GB  (slide 119: 'around 40 GB')")

opt = kv_bytes_per_token(layers=40, kv_heads=40, head_dim=128)
print(f"OPT-13B per token: {opt / 1024:.0f} KiB;  one 2,048-token request: {2048 * opt / 2**30:.2f} GiB")

## 2 · Real models, read from their configs
Every model on the Hub ships a `config.json` with the numbers we need. MLA (DeepSeek-V2/V3, slide 32) is different: per token and layer it
caches one compressed latent of size `kv_lora_rank` plus one shared rotary key of size `qk_rope_head_dim`, instead of keys and values
for every head. (LLaMA-2-70B is gated on the Hub, but Qwen2.5-72B has exactly the same attention shape: 80 layers, 64 query heads,
8 KV heads of size 128.)

In [ ]:
MODELS = ["facebook/opt-13b", "mistralai/Mistral-7B-v0.1", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen3-8B",
          "Qwen/Qwen2.5-72B-Instruct", "deepseek-ai/DeepSeek-V2", "deepseek-ai/DeepSeek-V3"]

def kv_from_config(c, bytes_per_value=2):
    L = c.get("num_hidden_layers") or c.get("n_layer")
    H = c.get("num_attention_heads") or c.get("n_head")
    hd = c.get("head_dim") or (c.get("hidden_size") or c.get("n_embd")) // H
    if c.get("kv_lora_rank"):                                   # MLA: latent + shared RoPE key
        kind, kv = "MLA", None
        actual = L * (c["kv_lora_rank"] + c["qk_rope_head_dim"]) * bytes_per_value
        hd = c.get("v_head_dim", hd)
    else:
        kv = c.get("num_key_value_heads") or H
        kind = "MHA" if kv == H else ("MQA" if kv == 1 else f"GQA-{kv}")
        actual = kv_bytes_per_token(L, kv, hd, bytes_per_value)
    as_mha = kv_bytes_per_token(L, H, hd, bytes_per_value)      # same model if every query head had its own K and V
    return dict(kind=kind, layers=L, q_heads=H, kv_heads=kv, head_dim=hd, actual=actual, as_mha=as_mha)

rows = {}
for m in MODELS:
    try:
        rows[m] = kv_from_config(json.load(open(hf_hub_download(m, "config.json"))))
    except Exception as e:
        print("skipped", m, type(e).__name__)

print(f"{'model':28s} {'attn':7s} {'layers':>6s} {'q/kv heads':>10s} {'KiB/token':>9s} {'GB @32K':>8s} {'MHA would be':>12s}")
for m, r in rows.items():
    kvh = r['kv_heads'] if r['kv_heads'] else '-'
    print(f"{m.split('/')[1]:28s} {r['kind']:7s} {r['layers']:6d} {str(r['q_heads']) + '/' + str(kvh):>10s} "
          f"{r['actual'] / 1024:9.0f} {32768 * r['actual'] / 1e9:8.2f} {r['as_mha'] / r['actual']:11.1f}x")

In [ ]:
names = [m.split("/")[1] for m in rows][::-1]
act = [r["actual"] / 1024 for r in rows.values()][::-1]
mha_ = [r["as_mha"] / 1024 for r in rows.values()][::-1]
fig, ax = plt.subplots(figsize=(9, 4.2))
y = range(len(names))
ax.barh([i + 0.2 for i in y], mha_, height=0.36, color="#c9c8c2", label="if every query head had its own K and V (MHA)")
ax.barh([i - 0.2 for i in y], act, height=0.36, color=BLUE, label="what the model actually caches")
for i, v in zip(y, act):
    ax.text(v * 1.08, i - 0.2, f"{v:.0f} KiB", va="center", fontsize=9, color=INK)
ax.set_yticks(list(y)); ax.set_yticklabels(names); ax.set_xscale("log"); ax.grid(axis="y", visible=False)
ax.set_xlabel("KV cache per token (KiB, BF16, log scale)"); ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.set_title("GQA and MLA shrink the KV cache by 4x to 57x", loc="left")
plt.tight_layout(); plt.show()

## 3 · How many users fit on one GPU?
The weights are shared by all requests; the cache is per request (slide 118). Take Qwen2.5-7B on one 80 GB GPU. Its weight size comes
from the Hub's `model.safetensors.index.json`; we ignore activations and other overheads, so real numbers are somewhat lower.

In [ ]:
M = "Qwen/Qwen2.5-7B-Instruct"
weights = json.load(open(hf_hub_download(M, "model.safetensors.index.json")))["metadata"]["total_size"]
r = rows[M]
free = 80e9 - weights
print(f"weights: {weights / 1e9:.1f} GB in BF16 -> {free / 1e9:.1f} GB left for the KV cache\n")
ctx = [1024, 2048, 4096, 8192, 16384, 32768]
users_gqa = [int(free // (n * r["actual"])) for n in ctx]
users_mha = [int(free // (n * r["as_mha"])) for n in ctx]
for n, a, b in zip(ctx, users_gqa, users_mha):
    print(f"context {n:6d}: {a:5d} concurrent requests with {r['kind']}   vs {b:4d} if it were MHA")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(ctx, users_gqa, color=BLUE, marker="o", ms=8); ax.plot(ctx, users_mha, color=ORANGE, marker="o", ms=8)
ax.text(ctx[-1], users_gqa[-1] * 1.25, r["kind"], color=INK, ha="right"); ax.text(ctx[-1], users_mha[-1] * 1.25, "MHA", color=INK, ha="right")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("context length (tokens)"); ax.set_ylabel("requests that fit")
ax.set_title("Qwen2.5-7B on one 80 GB GPU", loc="left"); plt.tight_layout(); plt.show()

## 4 · Check the formula against a real cache
SmolLM2-135M has 30 layers and 9 query heads of size 64, but only **3 KV heads** (GQA). We run a prompt through it and add up the bytes of
the cache the model returns. The tensors have shape `[batch, KV heads, tokens, head dim]` — look for the 3.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
SMOL = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(SMOL)
model = AutoModelForCausalLM.from_pretrained(SMOL, dtype=torch.float32).eval()
cfg = model.config

def cache_tensors(pkv):
    """(key, value) pairs per layer, for old and new versions of transformers."""
    if hasattr(pkv, "layers"):
        return [(l.keys, l.values) for l in pkv.layers]
    if hasattr(pkv, "key_cache"):
        return list(zip(pkv.key_cache, pkv.value_cache))
    return [tuple(x) for x in pkv]

prompt = ("Large language models are expensive to run. Every time a chatbot writes an answer, it produces one token at a time, "
          "and for each new token the model has to look back at everything that came before: the system prompt, the user's question "
          "and all the tokens it has already written. Engineers who build serving systems therefore care a lot about memory. "
          "The weights of the model are shared by all users, but each conversation keeps its own record of the past. "
          "In this short note we explain where that memory goes and how modern systems keep it small. The first thing to know is that")
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    out = model(ids, use_cache=True)
pairs = cache_tensors(out.past_key_values)
k0 = pairs[0][0]
print("layers in cache:", len(pairs), "  key tensor shape of layer 0:", tuple(k0.shape))
measured = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in pairs)
head_dim = cfg.hidden_size // cfg.num_attention_heads
predicted = ids.shape[1] * kv_bytes_per_token(cfg.num_hidden_layers, cfg.num_key_value_heads, head_dim, bytes_per_value=4)
print(f"{ids.shape[1]} tokens: measured {measured:,} bytes, formula {predicted:,} bytes (FP32 = 4 bytes per number)")
print(f"with all {cfg.num_attention_heads} heads (MHA) it would be {predicted * cfg.num_attention_heads // cfg.num_key_value_heads:,} bytes")

## 5 · Why cache at all?
Without a cache, every new token re-runs the whole sequence through the model; with it, each step only processes the newest token and
reads the stored keys and values (slide 117). We write the greedy decoding loop by hand so you can see both versions and time every step.

In [ ]:
@torch.no_grad()
def greedy(prompt_ids, n_new, use_cache):
    x, past, times = prompt_ids, None, []
    for _ in range(n_new):
        t0 = time.perf_counter()
        if use_cache and past is not None:
            o = model(x[:, -1:], past_key_values=past, use_cache=True)   # only the new token
        else:
            o = model(x, use_cache=use_cache)                            # the whole sequence again
        past = o.past_key_values if use_cache else None
        x = torch.cat([x, o.logits[:, -1].argmax(-1, keepdim=True)], dim=1)
        times.append(time.perf_counter() - t0)
    return x, times

N_NEW = 100
x_c, t_c = greedy(ids, N_NEW, use_cache=True)
x_n, t_n = greedy(ids, N_NEW, use_cache=False)
same = (x_c[0, ids.shape[1]:] == x_n[0, ids.shape[1]:]).float().mean().item()
print(f"with cache {sum(t_c):.1f} s, without {sum(t_n):.1f} s  -> {sum(t_n) / sum(t_c):.1f}x faster; same tokens: {same:.0%}")
print("continuation:", tok.decode(x_c[0, ids.shape[1]:ids.shape[1] + 40]))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot([t * 1000 for t in t_n], color=ORANGE); ax.plot([t * 1000 for t in t_c], color=BLUE)
ax.text(N_NEW - 1, t_n[-1] * 1000 * 1.05, "no cache", ha="right", va="bottom"); ax.text(N_NEW - 1, t_c[-1] * 1000 * 1.3, "KV cache", ha="right", va="bottom")
ax.set_xlabel("generation step"); ax.set_ylabel("ms per step"); ax.set_ylim(bottom=0)
ax.set_title(f"SmolLM2-135M on CPU, prompt of {ids.shape[1]} tokens", loc="left"); plt.tight_layout(); plt.show()

## 6 · What does sharing keys and values cost? MHA → GQA → MQA in GPT-2
The GQA paper (Ainslie et al., 2023) builds a grouped-query model from an existing multi-head checkpoint by **mean-pooling** the key and
value heads in each group, then briefly continues training ("uptraining"). Here we do only the first step on GPT-2 small (12 heads) and
measure perplexity on a slice of WikiText-2. With 12 groups nothing changes (MHA); with 1 group all heads share one K and one V (MQA).
Every query head stays different, so the cache would shrink by 12 / groups.

In [ ]:
import requests, re
# WikiText-2 test set (word-level version from the PyTorch examples repo); "<unk>" marks rare words.
URL = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/test.txt"
raw = requests.get(URL, timeout=60).text
text = " ".join(l.strip() for l in raw.splitlines() if l.strip() and not l.strip().startswith("="))
for a, b in [(" @-@ ", "-"), (" @,@ ", ","), (" @.@ ", "."), (" ,", ","), (" .", "."), (" ;", ";"), (" :", ":"), (" 's", "'s"), ("( ", "("), (" )", ")")]:
    text = text.replace(a, b)
print(len(text.split()), "words; first 300 characters:\n", text[:300])

In [ ]:
G2 = "openai-community/gpt2"
gtok = AutoTokenizer.from_pretrained(G2)
gpt2 = AutoModelForCausalLM.from_pretrained(G2, dtype=torch.float32).eval()
enc = gtok(text, return_tensors="pt").input_ids[:, :2048]

@torch.no_grad()
def perplexity(m, enc, window=1024):
    nll, n = 0.0, 0
    for i in range(0, enc.shape[1] - 1, window):
        chunk = enc[:, i:i + window]
        if chunk.shape[1] < 2: break
        loss = m(chunk, labels=chunk).loss
        nll += loss.item() * (chunk.shape[1] - 1); n += chunk.shape[1] - 1
    return float(torch.exp(torch.tensor(nll / n)))

D, H = gpt2.config.n_embd, gpt2.config.n_head; hd = D // H
backup = [(b.attn.c_attn.weight.data.clone(), b.attn.c_attn.bias.data.clone()) for b in gpt2.transformer.h]

def set_kv_groups(groups):
    """Mean-pool GPT-2's K and V heads within each group (c_attn packs Q | K | V along its output dim)."""
    size = H // groups
    for blk, (W0, b0) in zip(gpt2.transformer.h, backup):
        W, b = W0.clone(), b0.clone()
        for off in (D, 2 * D):                                   # the K block, then the V block
            w = W[:, off:off + D].reshape(D, groups, size, hd).mean(2, keepdim=True).expand(D, groups, size, hd)
            W[:, off:off + D] = w.reshape(D, D)
            bb = b[off:off + D].reshape(groups, size, hd).mean(1, keepdim=True).expand(groups, size, hd)
            b[off:off + D] = bb.reshape(D)
        blk.attn.c_attn.weight.data.copy_(W); blk.attn.c_attn.bias.data.copy_(b)

results = {}
for g in [12, 6, 4, 3, 2, 1]:
    set_kv_groups(g)
    results[g] = perplexity(gpt2, enc)
    label = "MHA" if g == H else ("MQA" if g == 1 else f"GQA-{g}")
    print(f"{label:6s} KV heads = {g:2d}  cache {H // g:2d}x smaller  perplexity {results[g]:8.1f}")
set_kv_groups(H)   # restore the original model

Even halving the number of KV heads ruins the model. Each head learned its own key space, matched to its own queries; averaging keys
from different heads breaks that pairing, and more sharing breaks it more. So sharing keys and values is not something you can switch on
after training. The GQA paper fixes it with a short **uptraining** phase (about 5% of the original pre-training compute), after which
GQA comes close to MHA quality at close to MQA speed. Models like Llama 2/3, Mistral and Qwen go one step further and are **trained with
GQA from the start**: they keep the cache savings from sections 2–3 without the quality loss.

## Exercises
1. How large is DeepSeek-V3's KV cache for a single 128K-token request? How large would it be with MHA? Use `kv_from_config`.
2. Many serving engines can store the cache in FP8 (`bytes_per_value=1`). How many more Qwen2.5-7B requests fit at 8K context?
3. In section 6, instead of mean-pooling, keep only the **first** head of every group. Is that better or worse? (The GQA paper compares both.)
4. In section 5, why does the time per step grow without the cache but stay flat with it? What would the total time look like for 2,000 new tokens?

**Next:** [Notebook 2 — quantize a small model and watch its perplexity](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_2_quantize_a_small_model.ipynb) · **Back to the [video lecture](https://nlp-course-cuhksz.github.io/lecture11.html)**